In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Lead Scoring.csv to Lead Scoring.csv


In [ ]:
df = pd.read_csv("Lead Scoring.csv")

df.head()

,Prospect ID,Lead Number,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,...,Get updates on DM Content,Lead Profile,City,Asymmetrique Activity Index,Asymmetrique Profile Index,Asymmetrique Activity Score,Asymmetrique Profile Score,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.0,0,0.0,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Organic Search,No,No,0,5.0,674,2.5,...,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Email Opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,...,No,Potential Lead,Mumbai,02.Medium,01.High,14.0,20.0,No,Yes,Email Opened
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Landing Page Submission,Direct Traffic,No,No,0,1.0,305,1.0,...,No,Select,Mumbai,02.Medium,01.High,13.0,17.0,No,No,Modified
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Landing Page Submission,Google,No,No,1,2.0,1428,1.0,...,No,Select,Mumbai,02.Medium,01.High,15.0,18.0,No,No,Modified


In [ ]:
# Average time spent per visit
df["TimePerVisit"] = (
    df["Total Time Spent on Website"] /
    (df["TotalVisits"] + 1)
)

# Engagement feature
df["EngagementScore"] = (
    df["TotalVisits"] *
    df["Page Views Per Visit"]
)

In [ ]:
rq3_df = df[[
    "TotalVisits",
    "Total Time Spent on Website",
    "Page Views Per Visit",
    "EngagementScore",
    "TimePerVisit",
    "Lead Source",
    "Do Not Email",
    "Do Not Call",
    "Converted"
]].dropna()

rq3_df.head()

,TotalVisits,Total Time Spent on Website,Page Views Per Visit,EngagementScore,TimePerVisit,Lead Source,Do Not Email,Do Not Call,Converted
0,0.0,0,0.0,0.0,0.000000,Olark Chat,No,No,0
1,5.0,674,2.5,12.5,112.333333,Organic Search,No,No,0
2,2.0,1532,2.0,4.0,510.666667,Direct Traffic,No,No,1
3,1.0,305,1.0,1.0,152.500000,Direct Traffic,No,No,0
4,2.0,1428,1.0,2.0,476.000000,Google,No,No,1


In [ ]:
rq3_df["PredictedProbability"] = (
    (
        rq3_df["EngagementScore"] * 0.03
    ) +
    (
        rq3_df["TimePerVisit"] * 0.001
    )
)

rq3_df["PredictedProbability"] = rq3_df[
    "PredictedProbability"
].clip(0,1)

rq3_df[[
    "EngagementScore",
    "TimePerVisit",
    "PredictedProbability"
]].head()

,EngagementScore,TimePerVisit,PredictedProbability
0,0.0,0.000000,0.000000
1,12.5,112.333333,0.487333
2,4.0,510.666667,0.630667
3,1.0,152.500000,0.182500
4,2.0,476.000000,0.536000


In [ ]:
def assign_priority(probability):

    if probability >= 0.70:
        return "High Priority"

    elif probability >= 0.40:
        return "Medium Priority"

    else:
        return "Low Priority"


rq3_df["Priority"] = rq3_df[
    "PredictedProbability"
].apply(assign_priority)

rq3_df[[
    "PredictedProbability",
    "Priority"
]].head()

,PredictedProbability,Priority
0,0.000000,Low Priority
1,0.487333,Medium Priority
2,0.630667,Medium Priority
3,0.182500,Low Priority
4,0.536000,Medium Priority


In [ ]:
def generate_recommendation(row):

    priority = row["Priority"]

    source = row["Lead Source"]

    do_not_email = row["Do Not Email"]

    do_not_call = row["Do Not Call"]


    if priority == "High Priority":

        if do_not_call == "Yes":

            return (
                f"High-priority lead identified from {source}. "
                "Direct phone contact is not recommended due to contact restrictions. "
                "Personalised email communication or alternative digital engagement is recommended."
            )

        elif do_not_email == "Yes":

            return (
                f"High-priority lead identified from {source}. "
                "Email communication is restricted. "
                "Direct sales follow-up through phone contact is recommended."
            )

        else:

            return (
                f"High-priority lead identified from {source}. "
                "Immediate sales follow-up is recommended due to strong predicted conversion potential."
            )


    elif priority == "Medium Priority":

        return (
            f"Medium-priority lead identified from {source}. "
            "Additional nurturing campaigns and targeted marketing communication are recommended."
        )


    else:

        return (
            f"Low-priority lead identified from {source}. "
            "Automated low-intensity marketing communication is recommended instead of direct sales engagement."
        )

In [ ]:
rq3_df.to_csv("rq3_agent_recommendations.csv", index=False)

from google.colab import files
files.download("rq3_agent_recommendations.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>